# Parte 1 - KPIs con Pandas

## 🎯 Contexto del Negocio

> Trabajas como Data Value Engineering en **"LearnHub"**, una plataforma de cursos online. Tu tarea es analizar el comportamiento de los usuarios y calcular KPIs clave usando pandas para la toma de decisiones estratégicas.
>

In [1]:
import pandas as pd
import numpy as np

# Configurar semilla para reproducibilidad
np.random.seed(42)

# Generar 100 registros
n_registros = 100

# Generar datos
data = {
    'usuario_id': range(1, n_registros + 1),

    'fecha_registro': pd.date_range('2023-01-01', periods=n_registros, freq='3D'),

    'fecha_primera_compra': [
        pd.Timestamp('2023-01-01') + pd.Timedelta(days=np.random.randint(1, 30))
        if np.random.random() > 0.25 else None
        for _ in range(n_registros)
    ],

    'fecha_ultima_actividad': pd.date_range('2024-10-01', periods=n_registros, freq='2D'),

    'plan': np.random.choice(
        ['Free', 'Basic', 'Standard', 'Premium'],
        n_registros,
        p=[0.25, 0.30, 0.25, 0.20]
    ),

    'precio_plan': None,  # Se calculará después según el plan

    'canal_adquisicion': np.random.choice(
        ['Google Ads', 'Facebook Ads', 'Instagram Ads', 'Organic', 'Referral', 'Email Marketing'],
        n_registros,
        p=[0.20, 0.15, 0.15, 0.25, 0.15, 0.10]
    ),

    'gasto_marketing': None,  # Se calculará después según el canal

    'cursos_completados': np.random.randint(0, 15, n_registros),

    'total_cursos_inscritos': None,  # Se calculará después

    'tiempo_plataforma_horas': np.random.uniform(0, 200, n_registros).round(2),

    'pais': np.random.choice(
        ['España', 'México', 'Argentina', 'Colombia', 'Chile', 'Perú', 'Estados Unidos'],
        n_registros,
        p=[0.20, 0.20, 0.15, 0.15, 0.10, 0.10, 0.10]
    ),

    'estado_suscripcion': np.random.choice(
        ['Activo', 'Cancelado', 'Pausado'],
        n_registros,
        p=[0.65, 0.25, 0.10]
    ),

    'numero_recomendaciones': np.random.randint(0, 8, n_registros),

    'certificados_obtenidos': None  # Se calculará después
}

df = pd.DataFrame(data)

# Asignar precios según el plan
precios = {'Free': 0, 'Basic': 29.99, 'Standard': 59.99, 'Premium': 99.99}
df['precio_plan'] = df['plan'].map(precios)

# Asignar gasto de marketing según el canal (solo para canales pagos)
gastos_canal = {
    'Google Ads': lambda: np.random.uniform(10, 25),
    'Facebook Ads': lambda: np.random.uniform(8, 20),
    'Instagram Ads': lambda: np.random.uniform(7, 18),
    'Organic': lambda: 0,
    'Referral': lambda: np.random.uniform(0, 5),
    'Email Marketing': lambda: np.random.uniform(2, 8)
}

df['gasto_marketing'] = df['canal_adquisicion'].apply(
    lambda x: round(gastos_canal[x](), 2)
)

# Cursos inscritos siempre >= cursos completados
df['total_cursos_inscritos'] = df['cursos_completados'] + np.random.randint(0, 10, n_registros)

# Certificados obtenidos (solo para cursos completados, no todos los cursos dan certificado)
df['certificados_obtenidos'] = df['cursos_completados'].apply(
    lambda x: np.random.randint(0, max(1, x + 1)) if x > 0 else 0
)

# Los usuarios Free no tienen fecha de compra ni pagan
df.loc[df['plan'] == 'Free', 'fecha_primera_compra'] = None

# Ajustar fechas de última actividad para usuarios cancelados (más antiguas)
mask_cancelados = df['estado_suscripcion'] == 'Cancelado'
df.loc[mask_cancelados, 'fecha_ultima_actividad'] = pd.date_range(
    '2024-06-01',
    periods=mask_cancelados.sum(),
    freq='3D'
)

print("Dataset creado exitosamente!")
print(f"Total de registros: {len(df)}")
print(f"\nPrimeros registros:")
print(df.head())
print(f"\nInformación del dataset:")
print(df.info())

Dataset creado exitosamente!
Total de registros: 100

Primeros registros:
   usuario_id fecha_registro fecha_primera_compra fecha_ultima_actividad  \
0           1     2023-01-01                  NaT             2024-10-01   
1           2     2023-01-04                  NaT             2024-10-03   
2           3     2023-01-07           2023-01-22             2024-10-05   
3           4     2023-01-10                  NaT             2024-10-07   
4           5     2023-01-13                  NaT             2024-10-09   

       plan  precio_plan canal_adquisicion  gasto_marketing  \
0      Free         0.00     Instagram Ads            11.15   
1  Standard        59.99           Organic             0.00   
2     Basic        29.99          Referral             2.31   
3      Free         0.00        Google Ads            14.52   
4     Basic        29.99   Email Marketing             6.49   

   cursos_completados  total_cursos_inscritos  tiempo_plataforma_horas  \
0               

## 🎯 Ejercicios

### **Ejercicio 1: Tasa de Conversión Global**

**Objetivo:** Calcular el porcentaje de usuarios registrados que se convirtieron de plan gratuito a plan de pago.

**Ecuación:**

```
Tasa de Conversión (%) = (Usuarios con Plan de Pago / Total de Usuarios Registrados) × 100
```

**Significado teórico:**
Este KPI mide la efectividad del modelo freemium de la plataforma. Una tasa de conversión alta indica que los usuarios perciben suficiente valor en la versión gratuita como para justificar el pago por funcionalidades premium. Típicamente, las plataformas freemium exitosas tienen tasas de conversión entre 2-5%.

In [ ]:
num_usr_pago = (df['plan'] != 'Free').sum()
total_usr = df.shape[0]
print(f'Usuarios de pago = {num_usr_pago}')
print(f'Usuarios totales = {total_usr}')
print(f'tasa = {((num_usr_pago/total_usr) * 100)} %')

Usuarios de pago = 82
Usuarios totales = 100
tasa = 82.0 %


### **Ejercicio 2: Tiempo Promedio de Conversión**

**Objetivo:** Para los usuarios que compraron un plan de pago, calcular el tiempo promedio (en días) que transcurrió entre su registro y su primera compra.

**Ecuación:**

```
Tiempo Promedio de Conversión (días) = Promedio(fecha_primera_compra - fecha_registro)

```

*Nota: Calcular solo para usuarios con fecha_primera_compra no nula*

**Significado teórico:**
Mide la velocidad del ciclo de conversión y la efectividad del funnel de ventas. Un tiempo menor indica que la propuesta de valor es clara y convincente. Este KPI ayuda a optimizar estrategias de nurturing y permite establecer el momento óptimo para campañas de conversión.

In [39]:
df_sin_fecha = df.dropna(subset=['fecha_primera_compra'])
tiempo_conversion = df_sin_fecha['fecha_registro'] - df_sin_fecha['fecha_primera_compra'] 
promedio_timedelta = tiempo_conversion.mean()
promedio_conversion_dias = promedio_timedelta.total_seconds() / (24 * 3600)

print(f"Total de Usuarios Convertidos (con fecha de compra): {len(df_sin_fecha)}")
print(f"Tiempo Promedio de Conversión (Timedelta): {promedio_timedelta}")
print(f"Tiempo Promedio de Conversión (en días): {promedio_conversion_dias:.2f} días")

Total de Usuarios Convertidos (con fecha de compra): 55
Tiempo Promedio de Conversión (Timedelta): 149 days 18:19:38.181818182
Tiempo Promedio de Conversión (en días): 149.76 días


### **Ejercicio 3: ARPU (Average Revenue Per User)**

**Objetivo:** Calcular el ingreso promedio por usuario considerando únicamente usuarios que tienen un plan de pago.

**Ecuación:**

```
ARPU = Suma Total de Precios de Planes / Número de Usuarios con Plan de Pago

```

**Significado teórico:**
Indica el valor monetario promedio que genera cada cliente de pago. Es fundamental para proyecciones financieras, evaluación de estrategias de pricing y comparación con el CAC. Un ARPU creciente puede indicar éxito en estrategias de upselling.

In [56]:
mascara_pago = df['precio_plan'] > 0
df_pago =df[mascara_pago]
suma_total_ingresos = df['precio_plan'].sum()
num_usr_pago = df_pago.shape[0]
ARPU = suma_total_ingresos / num_usr_pago

print(f"Ingresos Totales (Solo clientes de pago): ${suma_total_ingresos:.2f}")
print(f"Total Clientes de Pago: {num_usr_pago}")
print(f"ARPU (Ingreso Promedio por Usuario de Pago): ${ARPU:.2f}")

Ingresos Totales (Solo clientes de pago): $4859.18
Total Clientes de Pago: 82
ARPU (Ingreso Promedio por Usuario de Pago): $59.26


### **Ejercicio 4: CAC por Canal de Adquisición**

**Objetivo:** Calcular el Costo de Adquisición de Cliente (CAC) para cada canal de marketing, considerando únicamente los usuarios que se convirtieron a planes de pago.

**Ecuación:**

```
CAC por Canal = (Gasto Total de Marketing en el Canal) / (Número de Usuarios de Pago Adquiridos por ese Canal)

```

**Significado teórico:**
Identifica la eficiencia de cada canal para adquirir clientes de pago (no solo registros). Permite optimizar el presupuesto de marketing invirtiendo más en canales con CAC bajo. Un CAC debe compararse siempre con el LTV para determinar la rentabilidad del canal.

In [85]:
for canal in df['canal_adquisicion'].unique():
    gasto = df.loc[df['canal_adquisicion'] == canal, 'gasto_marketing'].sum()
    total_usr = df.loc[df['canal_adquisicion'] == canal].shape[0]
    CAC = gasto - total_usr

    print(f'CAC {canal} = {round(CAC, 2)}')

CAC Instagram Ads = 195.65
CAC Organic = -21.0
CAC Referral = 10.73
CAC Google Ads = 448.03
CAC Email Marketing = 50.19
CAC Facebook Ads = 244.66


### **Ejercicio 5: Distribución de Ingresos por Plan**

**Objetivo:** Calcular qué porcentaje de los ingresos totales representa cada tipo de plan (Basic, Standard, Premium).

**Ecuación:**

```
Participación del Plan (%) = (Ingresos del Plan Específico / Ingresos Totales) × 100

donde:
Ingresos del Plan = Cantidad de Usuarios del Plan × Precio del Plan

```

**Significado teórico:**
Muestra la distribución de ingresos entre los diferentes tiers de producto. Ayuda a identificar qué plan es el motor de ingresos y cuál podría necesitar optimización. Esta información es crucial para decisiones sobre desarrollo de producto y estrategias de pricing.

In [136]:
ingresos_totales = 0
ingresos_por_plan = {}

for plan in df['plan'].unique():
    if plan != 'Free':
        total_usr = df.loc[df['plan'] == plan].shape[0]
        precio_plan = df.loc[df['plan'] == plan, 'precio_plan'].iloc[0]
        ingresos_plan = total_usr * precio_plan
        ingresos_totales += ingresos_plan
        ingresos_por_plan[plan] = ingresos_plan
        print(f'Ingresos totales por plan {plan} = {round(ingresos_plan, 2)}')
print(f'Ingresos totales {ingresos_totales}')

for plan, ingresos in ingresos_por_plan.items():
    participacion = (ingresos / ingresos_totales) * 100
    print(f'Participacion {plan}: {round(participacion, 2)} %')

Ingresos totales por plan Standard = 1439.76
Ingresos totales por plan Basic = 1019.66
Ingresos totales por plan Premium = 2399.76
Ingresos totales 4859.18
Participacion Standard: 29.63 %
Participacion Basic: 20.98 %
Participacion Premium: 49.39 %


### **Ejercicio 6: Tasa de Finalización de Cursos (Completion Rate)**

**Objetivo:** Calcular el porcentaje promedio de cursos que los usuarios completan respecto a los cursos en los que se inscriben.

**Ecuación:**

```
Completion Rate (%) = Promedio(cursos_completados / total_cursos_inscritos) × 100

```

*Nota: Excluir usuarios con 0 cursos inscritos para evitar divisiones por cero*

**Significado teórico:**
Mide el nivel de engagement y la calidad percibida del contenido. Una tasa alta indica que los usuarios encuentran valor en los cursos y se mantienen comprometidos. Una tasa baja puede señalar problemas de calidad del contenido, dificultad inadecuada o falta de motivación.

In [ ]:
df_con_cursos = df[df['total_cursos_inscritos'] > 0]
completion_rate_individual = df_con_cursos['cursos_completados'] / df_con_cursos['total_cursos_inscritos']
promedio = (completion_rate_individual.mean()) * 100
print(f'El promedio de cursos completados = {round(promedio, 2)}%')

El promedio de cursos completados = 59.8%


### **Ejercicio 7: ROI por Canal de Marketing**

**Objetivo:** Calcular el Retorno de Inversión para cada canal de adquisición considerando los ingresos generados vs. el gasto en marketing.

**Ecuación:**

```
Ingresos por Canal = Suma(precio_plan) de usuarios adquiridos por ese canal
Gasto por Canal = Suma(gasto_marketing) del canal
ROI por Canal (%) = ((Ingresos por Canal - Gasto por Canal) / Gasto por Canal) × 100

```

**Significado teórico:**
Determina la rentabilidad real de cada canal de marketing. Un ROI positivo indica que el canal es rentable; un ROI superior al 200% generalmente se considera excelente en marketing digital. Este KPI es crítico para la asignación eficiente del presupuesto de marketing.

In [204]:
roi_canales = {}

for canal in df['canal_adquisicion'].unique():
        df_canal = df[df['canal_adquisicion'] == canal]
        ingreso_canal = df_canal['precio_plan'].sum()
        gasto_canal = df_canal['gasto_marketing'].sum()
        total_usr = df_canal.shape[0]
        if gasto_canal > 0:
                roi = ((ingreso_canal - gasto_canal) / gasto_canal) * 100
                roi_canales[canal] = roi
                print(f'{canal}: {round(roi, 2)} %')
        else:
                print(f'{canal}: N/A (sin gasto en marketing)')
        print(f'- Increso por {canal}: {round(ingreso_canal, 2)}€ / Usuarios de {canal}: {total_usr}')
        print(f'- Gasto de {canal}: {round(gasto_canal, 2)}€')
        print(f'==================================================')


for canal, roi in roi_canales.items():
        print(f'{canal}: {round(roi, 2)} %')


Instagram Ads: 259.03 %
- Increso por Instagram Ads: 759.88€ / Usuarios de Instagram Ads: 16
- Gasto de Instagram Ads: 211.65€
Organic: N/A (sin gasto en marketing)
- Increso por Organic: 1039.82€ / Usuarios de Organic: 21
- Gasto de Organic: 0.0€
Referral: 2129.75 %
- Increso por Referral: 439.93€ / Usuarios de Referral: 9
- Gasto de Referral: 19.73€
Google Ads: 152.57 %
- Increso por Google Ads: 1199.8€ / Usuarios de Google Ads: 27
- Gasto de Google Ads: 475.03€
Email Marketing: 541.87 %
- Increso por Email Marketing: 379.92€ / Usuarios de Email Marketing: 9
- Gasto de Email Marketing: 59.19€
Facebook Ads: 295.88 %
- Increso por Facebook Ads: 1039.83€ / Usuarios de Facebook Ads: 18
- Gasto de Facebook Ads: 262.66€
Instagram Ads: 259.03 %
Referral: 2129.75 %
Google Ads: 152.57 %
Email Marketing: 541.87 %
Facebook Ads: 295.88 %


### **Ejercicio 9: Engagement Score por País**

**Objetivo:** Crear un score de engagement por país combinando múltiples métricas: cursos completados, tiempo en plataforma y certificados obtenidos.

**Ecuación:**

```
Engagement Score = Promedio por País de:
  (cursos_completados × 0.4) +
  (tiempo_plataforma_horas / 10 × 0.3) +
  (certificados_obtenidos × 0.3)

```

**Significado teórico:**
Un score compuesto permite evaluar el nivel de compromiso de usuarios en diferentes mercados geográficos. Los pesos asignados reflejan la importancia relativa de cada métrica (completar cursos es el indicador más fuerte de engagement). Este KPI ayuda a identificar mercados con mejor product-market fit y a personalizar estrategias por región.

In [ ]:
paises = {}

for pais in df['pais'].unique():
    df_pais = df[df['pais'] == pais]
    cursos_compl_pais = df_pais['cursos_completados'].mean()
    tiempo_horas_pais = df_pais['tiempo_plataforma_horas'].mean()
    certs_por_pais = df_pais['certificados_obtenidos'].mean()
    engagment_score = ((cursos_compl_pais * 0.4) + ((tiempo_horas_pais / 10)*0.3) + (certs_por_pais * 0.3))
    paises[pais] = engagment_score
    print(f'Cursos completados de {pais}: {round(cursos_compl_pais, 2)}')
    print(f'Horas en plataforma de {pais}: {round(tiempo_horas_pais, 2)}')
    print(f'Certificados obtenidos en {pais}: {round(certs_por_pais, 2)}')
    print('=====================================================')
    
for pais, engagment_score in paises.items():
    print(f'{pais}: {round(engagment_score, 2)}')

Cursos completados de Perú: 7.31
Horas en plataforma de Perú: 71.2
Certificados obtenidos en Perú: 2.77
Cursos completados de Argentina: 8.0
Horas en plataforma de Argentina: 126.15
Certificados obtenidos en Argentina: 3.73
Cursos completados de Colombia: 6.0
Horas en plataforma de Colombia: 107.4
Certificados obtenidos en Colombia: 3.06
Cursos completados de México: 7.29
Horas en plataforma de México: 127.97
Certificados obtenidos en México: 4.24
Cursos completados de España: 6.0
Horas en plataforma de España: 100.14
Certificados obtenidos en España: 2.59
Cursos completados de Estados Unidos: 6.27
Horas en plataforma de Estados Unidos: 121.95
Certificados obtenidos en Estados Unidos: 3.73
Cursos completados de Chile: 5.5
Horas en plataforma de Chile: 46.76
Certificados obtenidos en Chile: 1.0
Perú: 5.89
Argentina: 8.1
Colombia: 6.54
México: 8.03
España: 6.18
Estados Unidos: 7.29
Chile: 3.9


### **Ejercicio 10: Viral Coefficient (K-Factor)**

**Objetivo:** Calcular el coeficiente viral estimado basado en el número de recomendaciones realizadas y la tasa de conversión de referidos.

**Ecuación:**

```
Promedio de Invitaciones por Usuario = Promedio(numero_recomendaciones)
Tasa de Conversión de Referidos = Usuarios del Canal "Referral" con Plan de Pago / Total de Usuarios del Canal "Referral"
K-Factor = Promedio de Invitaciones × Tasa de Conversión de Referidos

```

**Significado teórico:**
El K-Factor mide el potencial de crecimiento viral de la plataforma. Un K > 1 indica crecimiento exponencial orgánico (cada usuario trae más de un usuario nuevo). Un K entre 0.5-1 es muy saludable y reduce significativamente el CAC. Este KPI es fundamental para plataformas que dependen del crecimiento orgánico y el marketing de referidos.

In [ ]:
promedio_inivitaciones = df['numero_recomendaciones'].mean()
usuarios_referral_total = df[df['canal_adquisicion'] == 'Referral'].shape[0]
usuarios_referral_pago = df[(df['canal_adquisicion'] == 'Referral') & (df['plan'] != 'Free')].shape[0]
tasa_conversion = usuarios_referral_pago / usuarios_referral_total
k_factor = promedio_inivitaciones * tasa_conversion
print(f' El coeficiente viral es: {round(k_factor, 2)}')
        

 El coeficiente viral es: 2.61
